In [2]:
import arcpy
import os

def select_identical_polygons(layer1_name, layer2_name, output_workspace, overlap_percentage=90):
    """
    Selects polygons from two layers that have a shape overlap of at least 
    a specified percentage.

    Args:
        layer1_name (str): The name of the first polygon layer.
        layer2_name (str): The name of the second polygon layer.
        output_workspace (str): The file geodatabase or folder path where
                                temporary and final data will be stored.
        overlap_percentage (int): The minimum required overlap percentage (e.g., 90).
    """

    # Set up the geoprocessing environment
    arcpy.env.overwriteOutput = True
    arcpy.env.workspace = output_workspace
    
    # Define temporary file names
    temp_intersect = os.path.join(arcpy.env.scratchGDB, "temp_intersect")
    temp_lyr1_join = os.path.join(arcpy.env.scratchGDB, "temp_lyr1_join")
    final_selection = "Final_Selected_Identical_Features"

    try:
        print("Step 1: Intersecting the two layers to find common areas...")
        # Perform the Intersect operation to find overlapping areas
        arcpy.analysis.Intersect([layer1_name, layer2_name], temp_intersect)

        print("Step 2: Calculating original and overlapping areas...")
        # Add and calculate an area field for each original layer
        original_area1_field = "OriginalArea1"
        original_area2_field = "OriginalArea2"
        arcpy.management.CalculateGeometryAttributes(layer1_name, [[original_area1_field, "AREA"]])
        arcpy.management.CalculateGeometryAttributes(layer2_name, [[original_area2_field, "AREA"]])

        # Add and calculate an area field for the intersected layer
        overlap_area_field = "OverlapArea"
        arcpy.management.CalculateGeometryAttributes(temp_intersect, [[overlap_area_field, "AREA"]])

        print("Step 3: Joining attributes to a single table...")
        # Join attributes from the original layers to the intersected output
        join_field1 = f"FID_{layer1_name}"
        join_field2 = f"FID_{layer2_name}"
        
        # Create in-memory layers for joining
        arcpy.management.MakeFeatureLayer(temp_intersect, "intersect_lyr")
        arcpy.management.MakeFeatureLayer(layer1_name, "lyr1")
        arcpy.management.MakeFeatureLayer(layer2_name, "lyr2")
        
        # Perform the joins
        arcpy.management.JoinField("intersect_lyr", join_field1, "lyr1", "OBJECTID", [original_area1_field])
        arcpy.management.JoinField("intersect_lyr", join_field2, "lyr2", "OBJECTID", [original_area2_field])
        
        print("Step 4: Calculating overlap percentages...")
        # Add new fields for the overlap percentages
        pct_overlap1_field = "PctOverlap1"
        pct_overlap2_field = "PctOverlap2"
        arcpy.management.AddField(temp_intersect, pct_overlap1_field, "DOUBLE")
        arcpy.management.AddField(temp_intersect, pct_overlap2_field, "DOUBLE")
        
        # Calculate the percentage of overlap relative to each original layer
        expression1 = f"(!{overlap_area_field}! / !{original_area1_field}!) * 100"
        expression2 = f"(!{overlap_area_field}! / !{original_area2_field}!) * 100"
        arcpy.management.CalculateField(temp_intersect, pct_overlap1_field, expression1)
        arcpy.management.CalculateField(temp_intersect, pct_overlap2_field, expression2)
        
        print("Step 5: Selecting the final features...")
        # Build the SQL query to select features with high overlap
        query = f"({pct_overlap1_field} >= {overlap_percentage} AND {pct_overlap2_field} >= {overlap_percentage})"
        
        # Create a new feature layer with the selected features
        arcpy.management.MakeFeatureLayer(temp_intersect, final_selection, query)
        
        print(f"\nSuccessfully created a new layer called '{final_selection}' containing features with >{overlap_percentage}% overlap.")
        
    except arcpy.ExecuteError:
        print(arcpy.GetMessages(2))
    except Exception as ex:
        print(ex.args[0])

# --- Example of how to use the function ---
if __name__ == "__main__":
    # --- CHANGE THESE VARIABLES TO MATCH YOUR DATA ---
    # The names of your input polygon layers
    layer1 = "zjd"
    layer2 = "zrz1"
    # The path to your output geodatabase or folder
    output_gdb = r"D:\CAD处理\CAD\不动产数据_1.gdb" 
    
    # Run the function
    select_identical_polygons(layer1, layer2, output_gdb)

Step 1: Intersecting the two layers to find common areas...
Step 2: Calculating original and overlapping areas...
Step 3: Joining attributes to a single table...
Step 4: Calculating overlap percentages...
Step 5: Selecting the final features...

Successfully created a new layer called 'Final_Selected_Identical_Features' containing features with >90% overlap.
